# 02 - Data Cleaning and Returns

This notebook loads raw adjusted price data from `data/raw/raw_prices.csv`, prepares weekly and monthly price series, calculates returns, calculates airline excess returns versus the S&P 500, and creates a Tableau-ready long-format dataset.

No regression analysis, oil sensitivity summary, or oil shock analysis is performed in this phase.

## Imports and path setup

Use project-relative paths so the notebook can run from either the repository root or the `notebooks/` folder.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RAW_PRICES_PATH = PROJECT_ROOT / 'data' / 'raw' / 'raw_prices.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TABLEAU_DIR = PROJECT_ROOT / 'data' / 'tableau'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLEAU_DIR.mkdir(parents=True, exist_ok=True)

WEEKLY_RETURNS_PATH = PROCESSED_DIR / 'weekly_returns.csv'
MONTHLY_RETURNS_PATH = PROCESSED_DIR / 'monthly_returns.csv'
WEEKLY_EXCESS_PATH = PROCESSED_DIR / 'weekly_excess_returns.csv'
MONTHLY_EXCESS_PATH = PROCESSED_DIR / 'monthly_excess_returns.csv'
TABLEAU_DATASET_PATH = TABLEAU_DIR / 'airline_oil_tableau_dataset.csv'

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw input path: {RAW_PRICES_PATH}')

Project root: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis
Raw input path: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis/data/raw/raw_prices.csv


## Load project configuration

Expected assets and airline classifications come from `src/config.py`.

In [2]:
from config import AIRLINE_CLASSIFICATION, TICKERS

expected_assets = list(TICKERS.keys())
airline_assets = list(AIRLINE_CLASSIFICATION.keys())
benchmark_asset = 'S&P 500'
oil_asset = 'Brent Oil'

print('Expected assets:', expected_assets)
print('Airline assets:', airline_assets)

Expected assets: ['Brent Oil', 'Ryanair', 'Lufthansa', 'Southwest', 'American Airlines', 'S&P 500']
Airline assets: ['Ryanair', 'Lufthansa', 'Southwest', 'American Airlines']


## Load and prepare raw price data

The date column is parsed as datetime, set as the index, sorted, and checked against the expected asset list. Missing values are reported here, but not blindly forward-filled before return calculation.

In [3]:
if not RAW_PRICES_PATH.exists():
    raise FileNotFoundError(f'Missing raw price file: {RAW_PRICES_PATH}')

raw_prices = pd.read_csv(RAW_PRICES_PATH)

if 'Date' not in raw_prices.columns:
    raise KeyError('Expected a Date column in raw_prices.csv')

raw_prices['Date'] = pd.to_datetime(raw_prices['Date'], errors='coerce')
if raw_prices['Date'].isna().any():
    bad_dates = raw_prices.loc[raw_prices['Date'].isna()].head()
    raise ValueError(f'Unparseable dates found: {bad_dates}')

missing_asset_columns = [asset for asset in expected_assets if asset not in raw_prices.columns]
if missing_asset_columns:
    raise KeyError(f'Missing expected asset columns: {missing_asset_columns}')

price_data = raw_prices[['Date'] + expected_assets].copy()
price_data = price_data.drop_duplicates(subset='Date', keep='last')
price_data = price_data.set_index('Date').sort_index()
price_data.index.name = 'Date'

# Ensure all price columns are numeric after CSV loading.
for column in expected_assets:
    price_data[column] = pd.to_numeric(price_data[column], errors='coerce')

print('Prepared price data shape:', price_data.shape)
display(price_data.head())
display(price_data.tail())

Prepared price data shape: (2181, 6)


,Brent Oil,Ryanair,Lufthansa,Southwest,American Airlines,S&P 500
Date,,,,,,
2018-01-02,66.570000,40.315563,17.618053,59.443661,51.647560,2695.810059
2018-01-03,67.839996,41.089314,17.978792,58.206196,51.014027,2713.060059
2018-01-04,68.070000,41.322971,17.874062,58.017868,51.335659,2723.989990
2018-01-05,67.620003,41.614090,17.606419,57.677120,51.316170,2743.149902
2018-01-08,67.779999,41.951168,17.780970,57.390167,50.809349,2747.709961


,Brent Oil,Ryanair,Lufthansa,Southwest,American Airlines,S&P 500
Date,,,,,,
2026-06-08,94.250000,56.330002,8.280,40.841652,13.60,7405.729980
2026-06-09,91.449997,58.000000,8.092,42.982304,14.09,7386.649902
2026-06-10,93.099998,55.750000,8.044,41.220001,13.42,7266.990234
2026-06-11,90.379997,59.389999,8.030,44.290001,14.65,7394.299805
2026-06-12,87.330002,60.330002,8.446,45.470001,14.98,7431.459961


## Raw missing-value check

Missing values are expected because assets trade on different exchanges with different holidays and calendars. They are reported before resampling.

In [4]:
raw_missing_summary = pd.DataFrame({
    'missing_values': price_data.isna().sum(),
    'missing_percent': (price_data.isna().mean() * 100).round(2),
})
display(raw_missing_summary)

,missing_values,missing_percent
Brent Oil,55,2.52
Ryanair,58,2.66
Lufthansa,37,1.70
Southwest,58,2.66
American Airlines,58,2.66
S&P 500,58,2.66


## Weekly and monthly price series

For each period, use the last available trading price in that period. This avoids blanket forward-filling daily gaps before calculating returns.

In [5]:
weekly_prices = price_data.resample('W-FRI').last().dropna(how='all')
monthly_prices = price_data.resample('ME').last().dropna(how='all')

print('Weekly prices shape:', weekly_prices.shape)
print('Monthly prices shape:', monthly_prices.shape)

display(weekly_prices.head())
display(monthly_prices.head())

Weekly prices shape: (441, 6)
Monthly prices shape: (102, 6)


,Brent Oil,Ryanair,Lufthansa,Southwest,American Airlines,S&P 500
Date,,,,,,
2018-01-05,67.620003,41.614090,17.606419,57.677120,51.316170,2743.149902
2018-01-12,69.870003,44.218800,17.699511,58.609718,56.988735,2786.239990
2018-01-19,68.610001,46.271923,17.239861,58.331726,56.589115,2810.300049
2018-01-26,70.519997,46.808186,16.803480,54.538597,51.725525,2872.870117
2018-02-02,68.580002,46.122536,16.338009,52.646515,50.780106,2762.129883


,Brent Oil,Ryanair,Lufthansa,Southwest,American Airlines,S&P 500
Date,,,,,,
2018-01-31,69.050003,47.003544,16.722025,54.520664,52.943863,2823.810059
2018-02-28,65.779999,46.448128,16.070366,51.866356,52.977322,2713.830078
2018-03-31,70.269997,47.057167,15.092875,51.474422,50.741043,2640.870117
2018-04-30,75.169998,42.123539,14.063024,47.475441,41.922886,2648.050049
2018-05-31,77.589996,44.398830,13.970357,45.902813,42.618015,2705.270020


## Calculate weekly and monthly returns

Returns are calculated from the resampled price series using the current period price divided by the prior period price minus one. No automatic filling is used.

In [6]:
weekly_returns = weekly_prices.div(weekly_prices.shift(1)).sub(1).dropna(how='all')
monthly_returns = monthly_prices.div(monthly_prices.shift(1)).sub(1).dropna(how='all')

print('Weekly returns shape:', weekly_returns.shape)
print('Monthly returns shape:', monthly_returns.shape)

display(weekly_returns.head())
display(monthly_returns.head())

Weekly returns shape: (440, 6)
Monthly returns shape: (101, 6)


,Brent Oil,Ryanair,Lufthansa,Southwest,American Airlines,S&P 500
Date,,,,,,
2018-01-12,0.033274,0.062592,0.005287,0.016169,0.110541,0.015708
2018-01-19,-0.018034,0.046431,-0.025970,-0.004743,-0.007012,0.008635
2018-01-26,0.027838,0.011589,-0.025312,-0.065027,-0.085946,0.022265
2018-02-02,-0.027510,-0.014648,-0.027701,-0.034693,-0.018278,-0.038547
2018-02-09,-0.084427,-0.039199,-0.064102,-0.054676,-0.070000,-0.051620


,Brent Oil,Ryanair,Lufthansa,Southwest,American Airlines,S&P 500
Date,,,,,,
2018-02-28,-0.047357,-0.011816,-0.038970,-0.048684,0.000632,-0.038947
2018-03-31,0.068258,0.013112,-0.060826,-0.007557,-0.042212,-0.026884
2018-04-30,0.069731,-0.104843,-0.068234,-0.077689,-0.173787,0.002719
2018-05-31,0.032194,0.054015,-0.006589,-0.033125,0.016581,0.021608
2018-06-30,0.023843,-0.014494,-0.113978,-0.000820,-0.128158,0.004842


## Calculate airline excess returns

Airline excess return equals airline return minus S&P 500 return for the same date and frequency.

In [7]:
if benchmark_asset not in weekly_returns.columns or benchmark_asset not in monthly_returns.columns:
    raise KeyError(f'Missing benchmark asset: {benchmark_asset}')

weekly_excess_returns = weekly_returns[airline_assets].subtract(weekly_returns[benchmark_asset], axis=0)
monthly_excess_returns = monthly_returns[airline_assets].subtract(monthly_returns[benchmark_asset], axis=0)
weekly_excess_returns.index.name = 'Date'
monthly_excess_returns.index.name = 'Date'

print('Weekly excess returns shape:', weekly_excess_returns.shape)
print('Monthly excess returns shape:', monthly_excess_returns.shape)

display(weekly_excess_returns.head())
display(monthly_excess_returns.head())

Weekly excess returns shape: (440, 4)
Monthly excess returns shape: (101, 4)


,Ryanair,Lufthansa,Southwest,American Airlines
Date,,,,
2018-01-12,0.046884,-0.010421,0.000461,0.094833
2018-01-19,0.037796,-0.034605,-0.013378,-0.015648
2018-01-26,-0.010675,-0.047577,-0.087291,-0.108210
2018-02-02,0.023899,0.010846,0.003854,0.020269
2018-02-09,0.012420,-0.012483,-0.003056,-0.018380


,Ryanair,Lufthansa,Southwest,American Airlines
Date,,,,
2018-02-28,0.027131,-0.000023,-0.009737,0.039579
2018-03-31,0.039997,-0.033941,0.019328,-0.015327
2018-04-30,-0.107562,-0.070953,-0.080407,-0.176506
2018-05-31,0.032406,-0.028198,-0.054733,-0.005027
2018-06-30,-0.019336,-0.118821,-0.005663,-0.133001


## Save return outputs

Save weekly returns, monthly returns, weekly airline excess returns, and monthly airline excess returns.

In [8]:
weekly_returns.to_csv(WEEKLY_RETURNS_PATH)
monthly_returns.to_csv(MONTHLY_RETURNS_PATH)
weekly_excess_returns.to_csv(WEEKLY_EXCESS_PATH)
monthly_excess_returns.to_csv(MONTHLY_EXCESS_PATH)

print(f'Saved weekly returns: {WEEKLY_RETURNS_PATH}')
print(f'Saved monthly returns: {MONTHLY_RETURNS_PATH}')
print(f'Saved weekly excess returns: {WEEKLY_EXCESS_PATH}')
print(f'Saved monthly excess returns: {MONTHLY_EXCESS_PATH}')

Saved weekly returns: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis/data/processed/weekly_returns.csv
Saved monthly returns: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis/data/processed/monthly_returns.csv
Saved weekly excess returns: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis/data/processed/weekly_excess_returns.csv
Saved monthly excess returns: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis/data/processed/monthly_excess_returns.csv


## Create Tableau-ready long-format dataset

The Tableau dataset includes only airline rows as the main asset rows. Each row also includes the corresponding Brent return and S&P 500 return for the same date and frequency.

In [9]:
def build_tableau_rows(returns_df, excess_df, frequency):
    rows = []
    for date, returns_row in returns_df.iterrows():
        sp500_return = returns_row.get(benchmark_asset)
        brent_return = returns_row.get(oil_asset)
        for airline in airline_assets:
            classification = AIRLINE_CLASSIFICATION[airline]
            rows.append({
                'date': date,
                'frequency': frequency,
                'asset': airline,
                'asset_type': 'Airline',
                'region': classification['region'],
                'business_model': classification['business_model'],
                'hedging_profile': classification['hedging_profile'],
                'return': returns_row.get(airline),
                'sp500_return': sp500_return,
                'excess_return': excess_df.loc[date, airline] if date in excess_df.index else pd.NA,
                'brent_return': brent_return,
            })
    return pd.DataFrame(rows)

weekly_tableau = build_tableau_rows(weekly_returns, weekly_excess_returns, 'Weekly')
monthly_tableau = build_tableau_rows(monthly_returns, monthly_excess_returns, 'Monthly')
tableau_dataset = pd.concat([weekly_tableau, monthly_tableau], ignore_index=True)

# For Tableau usability, remove rows where the core return fields are missing.
core_return_columns = ['return', 'sp500_return', 'excess_return', 'brent_return']
tableau_missing_before_drop = tableau_dataset[core_return_columns].isna().sum()
tableau_dataset = tableau_dataset.dropna(subset=core_return_columns).copy()
tableau_dataset['date'] = pd.to_datetime(tableau_dataset['date'])
tableau_dataset = tableau_dataset.sort_values(['frequency', 'date', 'asset']).reset_index(drop=True)

tableau_dataset.to_csv(TABLEAU_DATASET_PATH, index=False)

print('Tableau dataset shape:', tableau_dataset.shape)
print('Rows dropped because of missing core return fields:', int(tableau_missing_before_drop.max() - tableau_dataset[core_return_columns].isna().sum().max()))
display(tableau_dataset.head())

Tableau dataset shape: (2164, 11)
Rows dropped because of missing core return fields: 0


,date,frequency,asset,asset_type,region,business_model,hedging_profile,return,sp500_return,excess_return,brent_return
0,2018-02-28,Monthly,American Airlines,Airline,United States,Legacy,Historically lower / limited hedging,0.000632,-0.038947,0.039579,-0.047357
1,2018-02-28,Monthly,Lufthansa,Airline,Europe,Legacy,Mixed / moderate hedging,-0.038970,-0.038947,-0.000023,-0.047357
2,2018-02-28,Monthly,Ryanair,Airline,Europe,Low-cost,Historically higher hedging,-0.011816,-0.038947,0.027131,-0.047357
3,2018-02-28,Monthly,Southwest,Airline,United States,Low-cost,Historically strong hedging,-0.048684,-0.038947,-0.009737,-0.047357
4,2018-03-31,Monthly,American Airlines,Airline,United States,Legacy,Historically lower / limited hedging,-0.042212,-0.026884,-0.015327,0.068258


## Missing-value summary after returns

Report missing values in the resampled prices, returns, excess returns, and final Tableau dataset.

In [10]:
missing_value_summary = {
    'raw_prices': price_data.isna().sum().astype(int).to_dict(),
    'weekly_prices': weekly_prices.isna().sum().astype(int).to_dict(),
    'monthly_prices': monthly_prices.isna().sum().astype(int).to_dict(),
    'weekly_returns': weekly_returns.isna().sum().astype(int).to_dict(),
    'monthly_returns': monthly_returns.isna().sum().astype(int).to_dict(),
    'weekly_excess_returns': weekly_excess_returns.isna().sum().astype(int).to_dict(),
    'monthly_excess_returns': monthly_excess_returns.isna().sum().astype(int).to_dict(),
    'tableau_dataset': tableau_dataset.isna().sum().astype(int).to_dict(),
}

for name, summary in missing_value_summary.items():
    print()
    print(name)
    print(summary)


raw_prices
{'Brent Oil': 55, 'Ryanair': 58, 'Lufthansa': 37, 'Southwest': 58, 'American Airlines': 58, 'S&P 500': 58}

weekly_prices
{'Brent Oil': 0, 'Ryanair': 0, 'Lufthansa': 0, 'Southwest': 0, 'American Airlines': 0, 'S&P 500': 0}

monthly_prices
{'Brent Oil': 0, 'Ryanair': 0, 'Lufthansa': 0, 'Southwest': 0, 'American Airlines': 0, 'S&P 500': 0}

weekly_returns
{'Brent Oil': 0, 'Ryanair': 0, 'Lufthansa': 0, 'Southwest': 0, 'American Airlines': 0, 'S&P 500': 0}

monthly_returns
{'Brent Oil': 0, 'Ryanair': 0, 'Lufthansa': 0, 'Southwest': 0, 'American Airlines': 0, 'S&P 500': 0}

weekly_excess_returns
{'Ryanair': 0, 'Lufthansa': 0, 'Southwest': 0, 'American Airlines': 0}

monthly_excess_returns
{'Ryanair': 0, 'Lufthansa': 0, 'Southwest': 0, 'American Airlines': 0}

tableau_dataset
{'date': 0, 'frequency': 0, 'asset': 0, 'asset_type': 0, 'region': 0, 'business_model': 0, 'hedging_profile': 0, 'return': 0, 'sp500_return': 0, 'excess_return': 0, 'brent_return': 0}


## Phase 3 validation

Validate that all output files exist, returns are not empty, expected assets are present, excess returns cover all four airlines, and the Tableau dataset has valid dates, all four airlines, and both frequencies.

In [11]:
output_paths = [
    WEEKLY_RETURNS_PATH,
    MONTHLY_RETURNS_PATH,
    WEEKLY_EXCESS_PATH,
    MONTHLY_EXCESS_PATH,
    TABLEAU_DATASET_PATH,
]

for path in output_paths:
    assert path.exists(), f'Missing output file: {path}'

assert not weekly_returns.empty, 'Weekly returns are empty.'
assert not monthly_returns.empty, 'Monthly returns are empty.'
assert list(weekly_returns.columns) == expected_assets, 'Weekly return columns do not match expected assets.'
assert list(monthly_returns.columns) == expected_assets, 'Monthly return columns do not match expected assets.'
assert list(weekly_excess_returns.columns) == airline_assets, 'Weekly excess return columns do not match airline assets.'
assert list(monthly_excess_returns.columns) == airline_assets, 'Monthly excess return columns do not match airline assets.'

saved_tableau = pd.read_csv(TABLEAU_DATASET_PATH, parse_dates=['date'])
required_tableau_columns = [
    'date', 'frequency', 'asset', 'asset_type', 'region', 'business_model',
    'hedging_profile', 'return', 'sp500_return', 'excess_return', 'brent_return'
]
assert list(saved_tableau.columns) == required_tableau_columns, 'Tableau dataset columns are incorrect.'
assert set(saved_tableau['asset']) == set(airline_assets), 'Tableau dataset does not contain all four airlines.'
assert set(saved_tableau['frequency']) == {'Weekly', 'Monthly'}, 'Tableau dataset does not contain both Weekly and Monthly frequencies.'
assert (saved_tableau['asset_type'] == 'Airline').all(), 'Tableau asset_type must be Airline for every row.'
assert saved_tableau['date'].notna().all(), 'Tableau dataset contains invalid dates.'

validation_summary = {
    'output_files_exist': {str(path.relative_to(PROJECT_ROOT)): path.exists() for path in output_paths},
    'weekly_returns_shape': weekly_returns.shape,
    'monthly_returns_shape': monthly_returns.shape,
    'weekly_excess_returns_shape': weekly_excess_returns.shape,
    'monthly_excess_returns_shape': monthly_excess_returns.shape,
    'tableau_dataset_shape': saved_tableau.shape,
    'tableau_frequencies': sorted(saved_tableau['frequency'].unique().tolist()),
    'tableau_assets': sorted(saved_tableau['asset'].unique().tolist()),
    'missing_value_summary': missing_value_summary,
}

validation_summary

{'output_files_exist': {'data/processed/weekly_returns.csv': True,
  'data/processed/monthly_returns.csv': True,
  'data/processed/weekly_excess_returns.csv': True,
  'data/processed/monthly_excess_returns.csv': True,
  'data/tableau/airline_oil_tableau_dataset.csv': True},
 'weekly_returns_shape': (440, 6),
 'monthly_returns_shape': (101, 6),
 'weekly_excess_returns_shape': (440, 4),
 'monthly_excess_returns_shape': (101, 4),
 'tableau_dataset_shape': (2164, 11),
 'tableau_frequencies': ['Monthly', 'Weekly'],
 'tableau_assets': ['American Airlines', 'Lufthansa', 'Ryanair', 'Southwest'],
 'missing_value_summary': {'raw_prices': {'Brent Oil': 55,
   'Ryanair': 58,
   'Lufthansa': 37,
   'Southwest': 58,
   'American Airlines': 58,
   'S&P 500': 58},
  'weekly_prices': {'Brent Oil': 0,
   'Ryanair': 0,
   'Lufthansa': 0,
   'Southwest': 0,
   'American Airlines': 0,
   'S&P 500': 0},
  'monthly_prices': {'Brent Oil': 0,
   'Ryanair': 0,
   'Lufthansa': 0,
   'Southwest': 0,
   'American 